# Simple RNN

## Load data

Uses full feature set from read_data to predict DKPrice with cross-validation.

In [ ]:
import pandas as pd

from Modules.read_data import read_data

PRICE_ZONE = "DK1"  # "DK1" or "DK2"
TRAIN_HOURS = 8760

(
    DK1_train,
    DK1_test,
    DK2_train,
    DK2_test,
    DK1_train_weather,
    DK1_test_weather,
    DK2_train_weather,
    DK2_test_weather
) = read_data("combined_data_cleaned_v5.csv")

if PRICE_ZONE == "DK1":
    dataset_train = DK1_train.copy()
    dataset_test = DK1_test.copy()
elif PRICE_ZONE == "DK2":
    dataset_train = DK2_train.copy()
    dataset_test = DK2_test.copy()
else:
    raise ValueError("PRICE_ZONE must be 'DK1' or 'DK2'.")

# Keep legacy variable names used by later cells in this notebook.
df = pd.concat([dataset_train, dataset_test], ignore_index=True).sort_values("Time").reset_index(drop=True)
target_time = pd.Timestamp("2024-01-01 00:00:00")
history = dataset_train.loc[dataset_train["Time"] < target_time].copy()
history = history.iloc[-TRAIN_HOURS:].copy()
prices = history["DKPrice"].astype(float).values.reshape(-1, 1)

print(f"Using zone: {PRICE_ZONE}")
print(f"Train shape: {dataset_train.shape}")
print(f"Test shape: {dataset_test.shape}")
print(f"Features used for modeling: {[c for c in dataset_train.columns if c != 'Time']}")

Train data: 8760 samples, 1 features


Test CUDA

In [3]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("CUDA DIAGNOSTICS")
print("\nBasic Info:")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {device}")

if torch.cuda.is_available():
    print(f"\nGPU Info:")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"cuDNN Version: {torch.backends.cudnn.version()}")
    print(f"Device Count: {torch.cuda.device_count()}")

    test_tensor = torch.randn(100, 100).to(device)
    print(f"Tensor on CUDA: {test_tensor.is_cuda}")

else:
    print("\n  Running on CPU - no CUDA available")

CUDA DIAGNOSTICS

Basic Info:
CUDA available: True
Device: cuda

GPU Info:
GPU Name: NVIDIA GeForce RTX 5060 Ti
CUDA Version: 12.8
cuDNN Version: 91002
Device Count: 1
Tensor on CUDA: True


## Hyperparameter search

### Helper functions

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from sklearn.base import BaseEstimator, RegressorMixin
from torch.utils.data import DataLoader, TensorDataset

def set_seed(seed: int) -> None:
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def smape_mean(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    denom = np.abs(y_true) + np.abs(y_pred)
    vals = np.where(denom == 0, 0.0, 200.0 * np.abs(y_pred - y_true) / denom)
    return float(np.mean(vals))

class TabularSimpleRNN(nn.Module):
    """RNN over per-sample feature vectors reshaped as a short sequence."""

    def __init__(self, hidden_size: int, layers: int):
        super().__init__()
        self.rnn = nn.RNN(
            input_size=1,
            hidden_size=hidden_size,
            num_layers=layers,
            batch_first=True,
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.rnn(x)
        return self.fc(out[:, -1, :])

class TorchRNNRegressor(BaseEstimator, RegressorMixin):
    """Scikit-learn style regressor so it works with run_cross_validation."""

    def __init__(
        self,
        hidden_size: int = 32,
        layers: int = 1,
        learning_rate: float = 1e-3,
        epochs: int = 40,
        batch_size: int = 64,
        random_state: int = 42,
    ):
        self.hidden_size = hidden_size
        self.layers = layers
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.batch_size = batch_size
        self.random_state = random_state

    def _to_tensor_sequence(self, X):
        X_np = np.asarray(X, dtype=np.float32)
        if X_np.ndim != 2:
            raise ValueError(f"Expected X with shape (n_samples, n_features), got {X_np.shape}.")
        return torch.tensor(X_np, dtype=torch.float32).unsqueeze(-1)

    def fit(self, X, y):
        set_seed(self.random_state)

        X_tensor = self._to_tensor_sequence(X)
        y_np = np.asarray(y, dtype=np.float32).reshape(-1, 1)
        y_tensor = torch.tensor(y_np, dtype=torch.float32)

        self.model_ = TabularSimpleRNN(
            hidden_size=int(self.hidden_size),
            layers=int(self.layers),
        )
        self.loss_fn_ = nn.MSELoss()
        self.optimizer_ = torch.optim.Adam(self.model_.parameters(), lr=float(self.learning_rate))

        dataset_local = TensorDataset(X_tensor, y_tensor)
        loader = DataLoader(dataset_local, batch_size=int(self.batch_size), shuffle=True)

        self.model_.train()
        for _ in range(int(self.epochs)):
            for X_batch, y_batch in loader:
                self.optimizer_.zero_grad()
                preds = self.model_(X_batch)
                loss = self.loss_fn_(preds, y_batch)
                loss.backward()
                self.optimizer_.step()

        return self

    def predict(self, X):
        X_tensor = self._to_tensor_sequence(X)
        self.model_.eval()
        with torch.no_grad():
            preds = self.model_(X_tensor).squeeze(-1).cpu().numpy()
        return preds

### Search

Search grid

In [ ]:
import numpy as np

param_grid = {
    "hidden_size": [16, 32, 64],
    "layers": [1, 2],
    "learning_rate": [0.001, 0.0005],
    "epochs": [30, 60],
    "batch_size": [32, 64],
}

print("Total combinations:", np.prod([len(v) for v in param_grid.values()]))

Total combinations:  324


Hyperparameter search

In [ ]:
# from Modules.Cross_Validation_runner import run_cross_validation
from Modules.Validation2 import run_cross_validation
import itertools
from pathlib import Path

import pandas as pd

split_setup = 2

num_combinations = np.prod([len(v) for v in param_grid.values()])
print(f"Total number of combinations to test: {num_combinations}")

param_names = list(param_grid.keys())
param_values = list(param_grid.values())
all_combinations = list(itertools.product(*param_values))

results = []
for comb_number, combination in enumerate(all_combinations, start=1):
    params = dict(zip(param_names, combination))
    print(f"\nCombination {comb_number}/{num_combinations}: {params}")

    model = TorchRNNRegressor(
        hidden_size=int(params["hidden_size"]),
        layers=int(params["layers"]),
        learning_rate=float(params["learning_rate"]),
        epochs=int(params["epochs"]),
        batch_size=int(params["batch_size"]),
        random_state=42,
    )

    combination_results = run_cross_validation(
        model=model,
        dataset=dataset_train,
        dk_zone=PRICE_ZONE,
        split_setup=split_setup,
        train_window=3 * 8760,
        val_window=1 * 8784,
        val_start="2024-01-01 00:00:00",
        predict_period=4 * 168,
        stride=13 * 168,
        use_scaler=True,
        print_fold_results=False,
        plot=False,
        rf_models=None,
        use_precomputed_feature_values=False,
        precomputed_feature_predictions=None,
        use_forecasted_history=True,
    )

    row = {
        **params,
        "price_zone": PRICE_ZONE,
        "train_window": "3 years",
        "val_start": "2024-01-01",
        "avg_smape": combination_results["overall_avg_weekly_smape"],
        "avg_weekly_rmse": combination_results["overall_avg_weekly_rmse"],
        "avg_weekly_mae": combination_results["overall_avg_weekly_mae"],
        "avg_weekly_smape": combination_results["overall_avg_weekly_smape"],
        "avg_daily_rmse": combination_results["overall_avg_daily_rmse"],
        "avg_daily_mae": combination_results["overall_avg_daily_mae"],
        "avg_daily_smape": combination_results["overall_avg_daily_smape"],
        "avg_smape_day_1": combination_results["avg_smape_day_1"],
        "avg_smape_day_2": combination_results["avg_smape_day_2"],
        "avg_smape_day_3": combination_results["avg_smape_day_3"],
        "avg_smape_day_4": combination_results["avg_smape_day_4"],
        "avg_smape_day_5": combination_results["avg_smape_day_5"],
        "avg_smape_day_6": combination_results["avg_smape_day_6"],
        "avg_smape_day_7": combination_results["avg_smape_day_7"],
    }
    results.append(row)

results_df = pd.DataFrame(results).sort_values("avg_smape")

project_root = Path.cwd()
while project_root.name != "Speciale_Kode" and project_root.parent != project_root:
    project_root = project_root.parent

output_folder = project_root / "Deep learners" / "Simple RNN"
output_folder.mkdir(parents=True, exist_ok=True)

base_filename = f"{PRICE_ZONE}_rnn_hyperparameter_search_results"
filename = output_folder / f"{base_filename}.csv"
counter = 1
while filename.exists():
    filename = output_folder / f"{base_filename}_{counter}.csv"
    counter += 1

results_df.to_csv(filename, index=False, decimal=",")
print(f"\nResults saved to: {filename}")
display(results_df.head(10))

Total combinations: 324
Eval chunks (derived): 18
[1/324] Testing: {'hidden_size': 16, 'SEQ_LEN': 24, 'learning_rate': 0.001, 'epochs': 20, 'batch_size': 16, 'layers': 1}
Time: 0.00 mins, estimated 0.00 min remaining
  Epoch 10/20, Loss: 0.000296
  Epoch 20/20, Loss: 0.000241
  Validation avg_weekly_smape: 50.80%
[2/324] Testing: {'hidden_size': 16, 'SEQ_LEN': 24, 'learning_rate': 0.001, 'epochs': 20, 'batch_size': 16, 'layers': 2}
Time: 0.26 mins, estimated 41.71 min remaining
  Epoch 10/20, Loss: 0.000265
  Epoch 20/20, Loss: 0.000232
  Validation avg_weekly_smape: 94.75%
[3/324] Testing: {'hidden_size': 16, 'SEQ_LEN': 24, 'learning_rate': 0.001, 'epochs': 20, 'batch_size': 32, 'layers': 1}
Time: 0.55 mins, estimated 59.23 min remaining
  Epoch 10/20, Loss: 0.000446
  Epoch 20/20, Loss: 0.000266
  Validation avg_weekly_smape: 86.51%
[4/324] Testing: {'hidden_size': 16, 'SEQ_LEN': 24, 'learning_rate': 0.001, 'epochs': 20, 'batch_size': 32, 'layers': 2}
Time: 0.70 mins, estimated 56.11

## Train final model

In [ ]:
import random
from pathlib import Path
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import MinMaxScaler
from torch.utils.data import DataLoader, TensorDataset

# =========================
# TRAIN FINAL MODEL
# =========================
# Set these manually before running the cell
FINAL_HIDDEN_SIZE = 32
FINAL_SEQ_LEN = 48
FINAL_LEARNING_RATE = 0.0005
FINAL_EPOCHS = 50
FINAL_BATCH_SIZE = 32
FINAL_LAYERS = 2

# Rebuild scaled data and sequences using the manual SEQ_LEN
SEQ_LEN = FINAL_SEQ_LEN
LEARNING_RATE = FINAL_LEARNING_RATE
EPOCHS = FINAL_EPOCHS

train_series = prices.astype(float)
scaler = MinMaxScaler()
prices_scaled = scaler.fit_transform(train_series)

X, y = [], []
for i in range(len(prices_scaled) - SEQ_LEN):
    X.append(prices_scaled[i:i + SEQ_LEN])
    y.append(prices_scaled[i + SEQ_LEN])

X = np.array(X)
y = np.array(y)
X = torch.tensor(X, dtype=torch.float32).to(device)
y = torch.tensor(y, dtype=torch.float32).to(device)

# Train the final model with the manually chosen architecture
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

model = TabularSimpleRNN(hidden_size=FINAL_HIDDEN_SIZE, layers=FINAL_LAYERS).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

dataset = TensorDataset(X, y)
loader = DataLoader(dataset, batch_size=FINAL_BATCH_SIZE, shuffle=True)

start_time = time.time()
for epoch in range(EPOCHS):
    model.train()
    epoch_losses = []
    for X_batch, y_batch in loader:
        optimizer.zero_grad()
        output = model(X_batch)
        loss = criterion(output, y_batch)
        loss.backward()
        optimizer.step()
        epoch_losses.append(loss.item())

    print(f"Epoch {epoch + 1}/{EPOCHS}, Loss: {np.mean(epoch_losses):.6f}, Time: {(time.time() - start_time)/60:.2f} min")

print(f"\nModel trained in {(time.time() - start_time)/60:.2f} minutes.")

Epoch 1/20, Loss: 0.062588
Epoch 2/20, Loss: 0.054383
Epoch 3/20, Loss: 0.046841
Epoch 4/20, Loss: 0.039962
Epoch 5/20, Loss: 0.033742
Epoch 6/20, Loss: 0.028170
Epoch 7/20, Loss: 0.023234
Epoch 8/20, Loss: 0.018915
Epoch 9/20, Loss: 0.015189
Epoch 10/20, Loss: 0.012029
Epoch 11/20, Loss: 0.009401
Epoch 12/20, Loss: 0.007268
Epoch 13/20, Loss: 0.005588
Epoch 14/20, Loss: 0.004317
Epoch 15/20, Loss: 0.003407
Epoch 16/20, Loss: 0.002810
Epoch 17/20, Loss: 0.002478
Epoch 18/20, Loss: 0.002361
Epoch 19/20, Loss: 0.002413
Epoch 20/20, Loss: 0.002589

Price zone: DK1
Predicted time: 2024-01-01 00:00:00
Last 5 prices before prediction: [260.100006 220.660004 213.729996 200.309998 126.660004]
Predicted DKPrice: 783.51


Evaluate model

In [ ]:
# =========================
# EVALUATE MODEL IN 168-HOUR BLOCKS
# =========================
from pathlib import Path

EVAL_START_OFFSET_HOURS = 0   # fx 0 = start ved TARGET_TIME, 24 = start 1 dag efter
EVAL_BLOCK_HOURS = 168        # hver prognoseblok er 168 timer
EVAL_BLOCKS = 1               # ændr denne hvis du vil evaluere flere blokke
EVAL_HOURS = EVAL_BLOCK_HOURS * EVAL_BLOCKS

eval_start_time = target_time + pd.Timedelta(hours=EVAL_START_OFFSET_HOURS)
eval_end_time = eval_start_time + pd.Timedelta(hours=EVAL_HOURS)

# Brug den fulde zoneserie fra df (som allerede er filtreret til PRICE_ZONE)
zone_df = df.sort_values("Time").reset_index(drop=True)

actual_eval = zone_df.loc[
    (zone_df["Time"] >= eval_start_time) & (zone_df["Time"] < eval_end_time),
    "DKPrice",
].astype(float).values

if len(actual_eval) < EVAL_HOURS:
    raise ValueError(
        f"Ikke nok fremtidige data fra {eval_start_time} til {eval_end_time}. "
        f"Har {len(actual_eval)} timer, men skal bruge {EVAL_HOURS}."
    )

actual_eval = actual_eval[:EVAL_HOURS]

def recursive_block_forecast(actual_series, series_before_start, block_hours, model, scaler, device):
    predictions = []
    block_smapes = []
    block_maes = []
    block_rmses = []
    day_smapes_by_position = [[] for _ in range(7)]
    daily_smapes_all = []

    for block_start in range(0, len(actual_series), block_hours):
        block_end = min(block_start + block_hours, len(actual_series))
        block_actual = actual_series[block_start:block_end]

        if block_start == 0:
            block_history = series_before_start[-SEQ_LEN:]
        else:
            block_history = actual_series[block_start - SEQ_LEN:block_start]

        if len(block_history) < SEQ_LEN:
            raise ValueError(
                f"Not enough true history to start block at hour {block_start}. "
                f"Need {SEQ_LEN}, got {len(block_history)}."
            )

        seq_scaled = scaler.transform(np.asarray(block_history).reshape(-1, 1))
        block_predictions = []

        model.eval()
        with torch.no_grad():
            for _ in range(len(block_actual)):
                x_in = torch.tensor(seq_scaled, dtype=torch.float32).unsqueeze(0).to(device)
                next_scaled = model(x_in).cpu().numpy()[0, 0]
                next_pred = scaler.inverse_transform(np.array([[next_scaled]]))[0, 0]
                block_predictions.append(next_pred)
                seq_scaled = np.vstack([seq_scaled[1:], [[next_scaled]]])

        block_predictions = np.array(block_predictions)
        predictions.extend(block_predictions.tolist())

        block_smape = smape_mean(block_actual, block_predictions)
        block_mae = float(np.mean(np.abs(block_actual - block_predictions)))
        block_rmse = float(np.sqrt(np.mean((block_actual - block_predictions) ** 2)))
        block_smapes.append(block_smape)
        block_maes.append(block_mae)
        block_rmses.append(block_rmse)

        for day_idx in range(7):
            day_start = day_idx * 24
            day_end = min(day_start + 24, len(block_actual))
            if day_start >= len(block_actual):
                break
            day_smape = smape_mean(block_actual[day_start:day_end], block_predictions[day_start:day_end])
            day_smapes_by_position[day_idx].append(day_smape)
            daily_smapes_all.append(day_smape)

    return np.array(predictions), block_smapes, block_maes, block_rmses, day_smapes_by_position, daily_smapes_all

pre_eval_series = zone_df.loc[zone_df["Time"] < eval_start_time, "DKPrice"].astype(float).values
if len(pre_eval_series) < SEQ_LEN:
    raise ValueError(
        f"Ikke nok historik før {eval_start_time}. "
        f"Har {len(pre_eval_series)} timer, skal bruge mindst {SEQ_LEN}."
    )

pred_eval, block_smapes, block_maes, block_rmses, day_smapes_by_position, daily_smapes_all = recursive_block_forecast(
    actual_series=actual_eval,
    series_before_start=pre_eval_series,
    block_hours=EVAL_BLOCK_HOURS,
    model=model,
    scaler=scaler,
    device=device,
)

avg_weekly_smape = float(np.mean(block_smapes)) if block_smapes else float("nan")
avg_mae = float(np.mean(block_maes)) if block_maes else float("nan")
avg_rmse = float(np.mean(block_rmses)) if block_rmses else float("nan")

avg_day_smapes = [
    float(np.mean(values)) if values else float("nan")
    for values in day_smapes_by_position
]

max_avg_daily_smape = float(np.max(daily_smapes_all)) if daily_smapes_all else float("nan")
min_avg_daily_smape = float(np.min(daily_smapes_all)) if daily_smapes_all else float("nan")

print("\n=========================")
print("BLOCKED RECURSIVE EVALUATION")
print("=========================")
print(f"Eval start: {eval_start_time}")
print(f"Eval end  : {eval_end_time}")
print(f"Hours     : {EVAL_HOURS}")
print(f"Block size: {EVAL_BLOCK_HOURS}")
print(f"Blocks    : {EVAL_BLOCKS}")
print(f"Avg SMAPE : {avg_weekly_smape:.2f}%")
print(f"Avg MAE   : {avg_mae:.4f}")
print(f"Avg RMSE  : {avg_rmse:.4f}")
print(f"Max daily SMAPE in window: {max_avg_daily_smape:.2f}%")
print(f"Min daily SMAPE in window: {min_avg_daily_smape:.2f}%")

for i, (smape_value, mae_value, rmse_value) in enumerate(zip(block_smapes, block_maes, block_rmses), start=1):
    print(f"Block {i}: SMAPE={smape_value:.2f}%, MAE={mae_value:.4f}, RMSE={rmse_value:.4f}")

for i, value in enumerate(avg_day_smapes, start=1):
    print(f"Avg SMAPE day {i}: {value:.2f}%")

# =========================
# SAVE RESULTS TO CSV
# =========================
train_start_time = history["Time"].min()
train_end_time = history["Time"].max()

results_row = {
    "price_zone": PRICE_ZONE,
    "seq_len": SEQ_LEN,
    "train_hours": TRAIN_HOURS,
    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "target_time": str(target_time),
    "train_start_time": str(train_start_time),
    "train_end_time": str(train_end_time),
    "validation_start_time": str(eval_start_time),
    "validation_end_time": str(eval_end_time),
    "validation_hours": EVAL_HOURS,
    "eval_start_offset_hours": EVAL_START_OFFSET_HOURS,
    "eval_block_hours": EVAL_BLOCK_HOURS,
    "eval_blocks": EVAL_BLOCKS,
    "avg_weekly_smape": float(avg_weekly_smape),
    "avg_weekly_mae": float(avg_mae),
    "avg_weekly_rmse": float(avg_rmse),
    "avg_daily_smape": float(np.mean(daily_smapes_all)),
    "max_avg_daily_smape_pct": float(max_avg_daily_smape),
    "min_avg_daily_smape_pct": float(min_avg_daily_smape),
}

for i, value in enumerate(avg_day_smapes, start=1):
    results_row[f"smape_day_{i}_pct"] = float(value)

for i, value in enumerate(block_smapes, start=1):
    results_row[f"block_{i}_smape_pct"] = float(value)
for i, value in enumerate(block_maes, start=1):
    results_row[f"block_{i}_mae"] = float(value)
for i, value in enumerate(block_rmses, start=1):
    results_row[f"block_{i}_rmse"] = float(value)

results_df = pd.DataFrame([results_row])

notebook_dir = Path.cwd()
file_index = 1
while True:
    csv_name = f"rnn_{PRICE_ZONE}_eval_{file_index}.csv"
    csv_path = notebook_dir / csv_name
    if not csv_path.exists():
        break
    file_index += 1

results_df.to_csv(csv_path, index=False, sep=";", decimal=",")
print(f"\nSaved evaluation CSV: {csv_path}")


SMAPE EVALUATION
Eval start: 2024-01-01 00:00:00
Eval end  : 2024-01-08 00:00:00
Hours     : 168
Avg daily SMAPE : 54.15%
Avg weekly SMAPE: 54.15%
SMAPE day 1: 115.07%
SMAPE day 2: 78.33%
SMAPE day 3: 89.38%
SMAPE day 4: 33.44%
SMAPE day 5: 18.63%
SMAPE day 6: 21.11%
SMAPE day 7: 23.12%

Saved evaluation CSV: c:\Users\n_and\OneDrive\Delt skrivebord\Data Science\Speciale\Energinet\Delte scripts\Speciale_Kode\Deep learners\Simple RNN\rnn_DK1_eval_5.csv
